# CosMx cell type enrichment by IHC classification group

**Goal:** Compare cell type composition between MHC II IHC-positive and IHC-negative
patient groups in the CosMx tumor microenvironment. Cell type fractions are computed
per patient and compared using Mann-Whitney U or t-test depending on normality.
Macrophage/Monocyte enrichment in MHC II positive patients is the primary finding.

**Input:** `outputs/processed/tumor_data_scored.h5ad` — CosMx AnnData with cell type
labels; `data/patient_mhc_ii_classifications.csv` — IHC-based MHC II patient groups.

**Output:** `outputs/figures/figure4/fig4a.pdf`;
`outputs/tables/figure4/cell_type_enrichment_stats.csv`

## Setup

In [ ]:
# standard libraries
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# single-cell analysis
import anndata as ad
import scanpy as sc

# ceiba utilities
from ceiba.plot_utils import sig_label

# figure settings
sns.set(font_scale=1.7)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42  # embed fonts in PDF — required by most journals
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# paper-wide color palettes — consistent across all analyses
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

classification_palette = {
    'class II positive': cmap_high_low[0],
    'class II negative': cmap_high_low[1],
}

In [ ]:
# load paths from central config
# collaborators: update this path to point to your local copy of paths_config.yml
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# inputs
tumor_scored_path = repo_root / cfg['outputs']['processed'] / 'tumor_data_scored.h5ad'
patient_class_path  = Path(cfg['datasets']['patient_classifications']['raw'])

# output paths — figure 4
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

# output paths — cell type enrichment panels go in figure 3 supplemental
fig_out   = fig_dir   / 'figure3'
supp_out  = fig_dir   / 'figure3-supp'
table_out = table_dir / 'figure3'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

## Load data

The scored CosMx AnnData and IHC patient classifications are loaded and merged.
Only MHC II positive and negative patients are retained for the primary comparison;
clonal and no-malignant-cells groups are excluded.

In [ ]:
# load CosMx AnnData with cell type labels
labeled_adata = ad.read_h5ad(tumor_scored_path)

# load and merge IHC patient classifications
patient_classifications = pd.read_csv(patient_class_path, index_col=0)
patient_classifications.loc['tonsil'] = ['Tonsil', 'Tonsil', 'Tonsil']
patient_classifications['patient'] = patient_classifications.index

labeled_adata.obs['patient'] = labeled_adata.obs['patient'].astype(str)
patient_classifications['patient'] = patient_classifications['patient'].astype(str)

labeled_adata.obs = pd.merge(
    labeled_adata.obs, patient_classifications, on='patient', how='left'
)

# rename for display
labeled_adata.obs['Assigned Cell Type'] = labeled_adata.obs['cell_type_assigned']

print(labeled_adata.obs['patient classification'].value_counts())

## Compute cell type fractions per patient

Cell type composition is computed as the percentage of each cell type per patient,
restricted to MHC II positive and negative groups. Each patient contributes one
data point per cell type — this patient-level aggregation is the appropriate
unit for statistical comparison given the nested structure of the data.

In [ ]:
# filter to MHC II pos and neg only
obs_classified = labeled_adata.obs[
    labeled_adata.obs['patient classification'].isin(['class II positive', 'class II negative'])
].copy()

# compute per-patient cell type fractions
grouped_counts = (
    obs_classified
    .groupby(['patient', 'cell_type_assigned'])
    .size()
    .unstack(fill_value=0)
)
grouped_percentages = grouped_counts.div(grouped_counts.sum(axis=1), axis=0) * 100

# add patient classification back
grouped_percentages['patient classification'] = (
    obs_classified.groupby('patient')['patient classification'].first()
)

# melt to long format for plotting
boxplot_data = (
    grouped_percentages
    .reset_index()
    .melt(
        id_vars=['patient', 'patient classification'],
        var_name='Cell Type',
        value_name='Percentage',
    )
)

print(f"Patients: {grouped_percentages.shape[0]}")
print(boxplot_data['Cell Type'].value_counts())

## Statistical testing

For each cell type, normality is assessed with Shapiro-Wilk. If both groups are
normal, a two-sample t-test is used; otherwise Mann-Whitney U. Results are saved
to `outputs/tables/` and used for significance annotation in the figure.

In [ ]:
from scipy.stats import shapiro, mannwhitneyu, ttest_ind

cell_types = [ct for ct in boxplot_data['Cell Type'].unique()]
stats_results = {}

for ct in cell_types:
    neg = boxplot_data.loc[
        (boxplot_data['Cell Type'] == ct) &
        (boxplot_data['patient classification'] == 'class II negative'), 'Percentage'
    ].dropna().values

    pos = boxplot_data.loc[
        (boxplot_data['Cell Type'] == ct) &
        (boxplot_data['patient classification'] == 'class II positive'), 'Percentage'
    ].dropna().values

    # test normality in both groups
    _, p_norm_neg = shapiro(neg)
    _, p_norm_pos = shapiro(pos)
    normal = (p_norm_neg > 0.05) and (p_norm_pos > 0.05)

    if normal:
        stat, pval = ttest_ind(neg, pos)
        test = 't-test'
    else:
        stat, pval = mannwhitneyu(neg, pos, alternative='two-sided')
        test = 'Mann-Whitney U'

    stats_results[ct] = {
        'Test': test,
        'Statistic': stat,
        'P-value': pval,
        'Normal Negative': p_norm_neg > 0.05,
        'Normal Positive': p_norm_pos > 0.05,
    }

# save results table
stats_df = pd.DataFrame.from_dict(stats_results, orient='index')
stats_df.to_csv(table_out / 'cell_type_enrichment_stats.csv')
print(stats_df)

In [ ]:
max_values = boxplot_data.groupby(['Cell Type', 'patient classification'])['Percentage'].max()

fig, ax = plt.subplots(figsize=(14, 8))

sns.boxplot(
    data=boxplot_data,
    x='Cell Type', y='Percentage',
    hue='patient classification',
    dodge=True, gap=0.1,
    palette=classification_palette, fill=False,
    ax=ax,
)

sns.swarmplot(
    data=boxplot_data,
    x='Cell Type', y='Percentage',
    hue='patient classification',
    dodge=True, alpha=0.5,
    edgecolor='k', linewidth=0.5,
    palette=classification_palette,
    ax=ax, legend=False,
)

# significance annotations using ceiba sig_label
for i, ct in enumerate(boxplot_data['Cell Type'].unique()):
    pval = stats_results[ct]['P-value']
    if pval < 0.05:
        max_y = max_values[ct].max() + 5
        ax.plot([i - 0.2, i + 0.2], [max_y, max_y], color='black', lw=1.5)
        ax.text(i, max_y + 1, sig_label(pval),
                ha='center', va='bottom', fontsize=20, color='black')

ax.set_xlabel('')
ax.set_ylabel('Percentage of total cells')
ax.legend(title='Patient Classification', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.spines[['top', 'right']].set_visible(False)
plt.savefig(supp_out / 'figS3_cell_type_enrichment.pdf', bbox_inches='tight')
plt.show()